In [9]:
# @title 1. Installation of Required Libraries
# This cell ensures a clean and stable environment by first uninstalling any existing versions
# of ultralytics and its core dependencies before reinstalling everything from scratch.
# This is the most robust way to avoid version conflicts.


# Reinstall all packages, letting ultralytics manage its own dependencies
!pip install -q ultralytics pandas numpy opencv-python-headless matplotlib seaborn albumentations kaggle kagglehub


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# @title 2. (Optional) Download Dataset from Kaggle
# This cell provides an option to download the dataset directly from Kaggle.
# If you already have the 'Dataset' folder, you can set DOWNLOAD_FROM_KAGGLE to False.

import os
import json
import zipfile
import shutil

DOWNLOAD_FROM_KAGGLE = False # @param {type:"boolean"}

if DOWNLOAD_FROM_KAGGLE:
    print("Attempting to download dataset from Kaggle...")
    
    # --- Setup Kaggle API Token ---
    api_token = {"username":"tomeratar","key":"481b7c26190e8c6481146d7acf863ff9"}
    
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as file:
        json.dump(api_token, file)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600) # Set correct permissions
    
    # --- Download and Unzip ---
    import kagglehub
    
    output_dir = "Dataset"
    if os.path.exists(output_dir):
        print(f"Removing existing '{output_dir}' directory...")
        shutil.rmtree(output_dir)

    print("Downloading dataset from goelyash/animal-dataset...")
    download_path = kagglehub.dataset_download("goelyash/animal-dataset")
    print(f"Dataset downloaded to: {download_path}")

    # --- Verification and Cleanup ---
    try:
        with zipfile.ZipFile(download_path, 'r') as zip_ref:
            # Get the list of files in the zip to verify against later
            file_list = zip_ref.namelist()
            total_files_in_zip = len(file_list)
            print(f"Archive contains {total_files_in_zip} files.")
            
            print(f"Unzipping dataset to '{output_dir}' folder...")
            zip_ref.extractall(output_dir)

        # Count the number of files actually extracted
        extracted_files_count = 0
        for root, dirs, files in os.walk(output_dir):
            extracted_files_count += len(files)
        
        print(f"Successfully extracted {extracted_files_count} files.")

        if extracted_files_count == total_files_in_zip and extracted_files_count > 0:
             print("Verification successful: Extracted file count matches archive file count.")
             print(f"Deleting downloaded zip file: {download_path}")
             os.remove(download_path)
        else:
            error_message = (f"ERROR: Dataset extraction failed. "
                           f"Expected {total_files_in_zip} files, but found {extracted_files_count}. "
                           "This might be due to a lack of disk space. Terminating the notebook run.")
            print(error_message)
            raise RuntimeError(error_message)

    except zipfile.BadZipFile:
        error_message = "ERROR: The downloaded file is not a valid zip archive or is corrupted. Please try the download again."
        print(error_message)
        raise RuntimeError(error_message)
    
    BASE_PATH = output_dir + "/"

else:
    print("Skipping Kaggle download.")
    BASE_PATH = "Dataset/"

if not os.path.exists(BASE_PATH):
     raise FileNotFoundError(f"The specified BASE_PATH '{BASE_PATH}' does not exist. Please check the path or enable the Kaggle download.")
else:
    print(f"Dataset base path set to: {BASE_PATH}")

Skipping Kaggle download.
Dataset base path set to: Dataset/


In [11]:
# @title 3. Importing Libraries and Initial Setup
import pandas as pd
import numpy as np
import cv2
from tqdm.notebook import tqdm
import yaml
from ultralytics import YOLO
import matplotlib.pyplot as plt

# --- Configuration ---
TRAIN_ANNOTATION_PATH = os.path.join(BASE_PATH, "Annotation/Train_annotation.csv")
VAL_ANNOTATION_PATH = os.path.join(BASE_PATH, "Annotation/Val_annotation.csv")
CLASS_NAME_PATH = os.path.join(BASE_PATH, "Class_name.csv")

# --- Define output directories ---
TRAIN_LABEL_DIR = "yolo_data/labels/train"
VAL_LABEL_DIR = "yolo_data/labels/val"
TRAIN_IMAGE_DIR = "yolo_data/images/train"
VAL_IMAGE_DIR = "yolo_data/images/val"

# --- Clean up and create directories ---
if os.path.exists("yolo_data"):
    shutil.rmtree("yolo_data")

os.makedirs(TRAIN_LABEL_DIR, exist_ok=True)
os.makedirs(VAL_LABEL_DIR, exist_ok=True)
os.makedirs(TRAIN_IMAGE_DIR, exist_ok=True)
os.makedirs(VAL_IMAGE_DIR, exist_ok=True)

print("Directories created successfully.")

Directories created successfully.


In [12]:
# @title 4. Define the List of Dangerous Animals (with Correct Capitalization)
DANGEROUS_ANIMALS_LOWERCASE = [
    'alligator', 'american alligator', 'anemone fish', 'barracouta', 'boa', 'cobra', 
    'cougar', 'crayfish', 'crocodile', 'diamondback', 'dungeness crab', 'electric eel', 'electric ray', 
    'elephant', 'gar', 'gila monster', 'grizzly bear', 'hammerhead', 'hyena', 'indian cobra', 
    'jaguar', 'jellyfish', 'king crab', 'king snake', 'komodo dragon', 'leopard', 'lion', 'lionfish', 'lizard', 
    'lobster', 'mackerel shark', 'piranha', 'puffer', 'python', 'rat', 'rattlesnake', 'ray', 
    'requiem shark', 'rhinoceros', 'rock crab', 'salamander', 'shark', 'sidewinder', 'snake', 
    'spiny lobster', 'stingray', 'sturgeon', 'tiger', 'tiger shark', 'vulture', 'warthog', 
    'wild boar', 'wolf'
]

class_df = pd.read_csv(CLASS_NAME_PATH)
# Strip whitespace from names to prevent matching errors
class_df['Class_name'] = class_df['Class_name'].str.strip()
correct_case_map = {name.lower(): name for name in class_df['Class_name']}

DANGEROUS_ANIMALS = [correct_case_map.get(animal.strip(), animal.strip()) for animal in DANGEROUS_ANIMALS_LOWERCASE]

dangerous_class_to_int = {name: i for i, name in enumerate(DANGEROUS_ANIMALS)}

print(f"Total number of dangerous classes to train on: {len(DANGEROUS_ANIMALS)}")

Total number of dangerous classes to train on: 53


In [13]:
# @title 5. Data Preprocessing Function
def process_annotations(csv_path, image_dir, label_dir, is_train=True):
    # The DtypeWarning is informational and can be ignored, but low_memory=False is good practice
    df = pd.read_csv(csv_path, low_memory=False)
    # Strip whitespace from all name columns to be safe
    for col in df.columns:
        if 'Name_' in col:
            df[col] = df[col].astype(str).str.strip()
            
    max_objects = 15 
    name_cols = [f'Name_{i}' for i in range(1, max_objects + 1)]
    xmin_cols = [f'Xmin_{i}' for i in range(1, max_objects + 1)]
    ymin_cols = [f'Ymin_{i}' for i in range(1, max_objects + 1)]
    xmax_cols = [f'Xmax_{i}' for i in range(1, max_objects + 1)]
    ymax_cols = [f'Ymax_{i}' for i in range(1, max_objects + 1)]
    
    images_copied_count = 0

    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc=f"Processing {'Train' if is_train else 'Val'} data"):
        if is_train:
            # FIX from last time: Using split('/')[0] for the directory
            class_id = row['File_location'].split('/')[0]
            src_img_path = os.path.join(BASE_PATH, 'Train', class_id, f"{row['File_location'].split('/')[-1]}.JPEG")
        else:
            src_img_path = os.path.join(BASE_PATH, 'val', 'val', f"{row['File_location'].split('/')[-1]}.JPEG")

        if not os.path.exists(src_img_path):
            continue
            
        img_width = row['Width']
        img_height = row['Height']
        yolo_annotations = []
        
        for i in range(max_objects):
            # NEW FIX: Check if the column exists before trying to access it
            if name_cols[i] in df.columns:
                class_name = row[name_cols[i]]
                if pd.isna(class_name) or class_name == '0' or class_name not in DANGEROUS_ANIMALS:
                    continue

                class_int = dangerous_class_to_int[class_name]
                xmin, ymin, xmax, ymax = row[xmin_cols[i]], row[ymin_cols[i]], row[xmax_cols[i]], row[ymax_cols[i]]
                
                box_width = (xmax - xmin) / img_width
                box_height = (ymax - ymin) / img_height
                center_x = (xmin + (xmax - xmin) / 2) / img_width
                center_y = (ymin + (ymax - ymin) / 2) / img_height
                yolo_annotations.append(f"{class_int} {center_x} {center_y} {box_width} {box_height}")

        if yolo_annotations:
            img_filename = os.path.basename(src_img_path)
            label_filename = os.path.splitext(img_filename)[0] + ".txt"
            with open(os.path.join(label_dir, label_filename), 'w') as f:
                f.write("\n".join(yolo_annotations))
            shutil.copy(src_img_path, os.path.join(image_dir, img_filename))
            images_copied_count += 1
    
    print(f"\nFinished processing. Total images copied: {images_copied_count}")
    return images_copied_count

print("Annotation processing function defined.")

Annotation processing function defined.


In [ ]:
# @title 6. Process Training and Validation Datasets
print("Starting to process the training data...")
train_count = process_annotations(TRAIN_ANNOTATION_PATH, TRAIN_IMAGE_DIR, TRAIN_LABEL_DIR, is_train=True)

print("\nStarting to process the validation data...")
val_count = process_annotations(VAL_ANNOTATION_PATH, VAL_IMAGE_DIR, VAL_LABEL_DIR, is_train=False)

if train_count == 0:
    error_message = "CRITICAL ERROR: No training images were copied. This means no matching animals were found in the training annotations. Please check the DANGEROUS_ANIMALS list and the annotation files. Halting execution."
    print(error_message)
    raise RuntimeError(error_message)

Starting to process the training data...


Processing Train data:   0%|          | 0/218060 [00:00<?, ?it/s]

In [ ]:
# @title 7. Create the `dataset.yaml` File
dataset_yaml_content = {
    'train': os.path.abspath(TRAIN_IMAGE_DIR),
    'val': os.path.abspath(VAL_IMAGE_DIR),
    'nc': len(DANGEROUS_ANIMALS),
    'names': DANGEROUS_ANIMALS
}

YAML_PATH = 'dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(dataset_yaml_content, f, default_flow_style=False)

print(f"'{YAML_PATH}' created successfully!")
print("\n--- YAML File Content ---")
with open(YAML_PATH, 'r') as f:
    print(f.read())
print("------------------------")

# 8. Augmentation Strategy
With ~1300 images per class, a strong augmentation strategy is crucial for robustness. The `ultralytics` library has powerful, built-in augmentations that are applied on-the-fly during training. We will use the default YOLO augmentation settings, which are highly optimized for performance.

In [ ]:
# @title 9. MAX SPEED Model Training (GPU Accelerated)

# This cell is optimized for the FASTEST possible training speed on a CUDA-enabled GPU.
# It uses mixed precision, a smaller image size, and optimized data handling.
# NOTE: Reducing image size improves speed but may lower accuracy for small objects.

import os
from ultralytics import YOLO

yolov10_versions = ['yolov10n.pt', 'yolov10s.pt', 'yolov10m.pt', 'yolov10l.pt', 'yolov10x.pt']

for version in yolov10_versions:
    print(f"\n{'='*20} TRAINING {version.upper()} {'='*20}")
    
    model = YOLO(version)
    run_name = f'yolo_dangerous_animal_detection_{version.split(".")[0]}'
    
    # Start training with parameters optimized for SPEED on a GPU
    model.train(
        data=YAML_PATH,
        epochs=300,
        patience=50,
        batch=-1,           # Automatically uses the largest batch size that fits in VRAM
        name=run_name,
        exist_ok=True,
        
        # --- GPU & MAX SPEED OPTIMIZATIONS ---
        device='cuda:0',    # 🚀 **THIS IS THE KEY CHANGE TO USE THE GPU**
        amp=True,           # Enables mixed precision (FP16) for a major speed boost on NVIDIA GPUs
        imgsz=416,          # Processes smaller images, which is much faster
        workers=8,          # Speeds up data loading from the CPU
        cache=True          # Loads the entire dataset into RAM to eliminate disk reads after epoch 1
    )
    print(f"--- Finished Training {version.upper()} ---")

print("\n\nAll YOLOv10 model versions have been trained.")

In [ ]:
# @title 10. Validation and Results for All Models
# This cell validates each trained model and collects their performance metrics.

yolov10_versions = ['yolov10n.pt', 'yolov10s.pt', 'yolov10m.pt', 'yolov10l.pt', 'yolov10x.pt']
all_metrics = []

for version in yolov10_versions:
    model_name_short = version.split('.')[0]
    run_name = f'yolo_dangerous_animal_detection_{model_name_short}'
    best_model_path = f'runs/detect/{run_name}/weights/best.pt'
    
    if os.path.exists(best_model_path):
        print(f"\n--- Validating {model_name_short.upper()} ---")
        model = YOLO(best_model_path)
        metrics = model.val()
        
        all_metrics.append({
            'Model': model_name_short.upper(),
            'mAP50-95': metrics.box.map,
            'mAP50': metrics.box.map50,
            'mAP75': metrics.box.map75
        })
    else:
        print(f"\nCould not find trained weights for {model_name_short.upper()} at {best_model_path}. Skipping validation.")

In [ ]:
# @title 11. Results Summary
# This cell displays a summary table comparing the performance of all trained models.

if all_metrics:
    results_df = pd.DataFrame(all_metrics)
    results_df = results_df.set_index('Model')
    print("\n--- Performance Comparison of YOLOv10 Models ---")
    print(results_df)
else:
    print("\nNo validation metrics were collected. Please ensure the training completed successfully.")

In [ ]:
# @title 12. Inference on a Sample Image
# Choose which model version you want to use for this inference test.
# Options: 'n', 's', 'm', 'l', 'x'
CHOSEN_MODEL_VERSION = 'l' # @param {type:"string"}

model_name_short = f'yolov10{CHOSEN_MODEL_VERSION}'
run_name = f'yolo_dangerous_animal_detection_{model_name_short}'
best_model_path = f'runs/detect/{run_name}/weights/best.pt'

if not os.path.exists(best_model_path):
    print(f"ERROR: Could not find weights for the chosen model '{model_name_short.upper()}'. Please check if it trained correctly.")
else:
    print(f"Loading model: {best_model_path}")
    model = YOLO(best_model_path)
    
    sample_image_path = None
    if os.path.exists(VAL_IMAGE_DIR) and len(os.listdir(VAL_IMAGE_DIR)) > 0:
        sample_image_path = os.path.join(VAL_IMAGE_DIR, os.listdir(VAL_IMAGE_DIR)[0])

    if sample_image_path:
        print(f"Running inference on: {sample_image_path}")
        results = model([sample_image_path])
        for r in results:
            im_array = r.plot()
            im_array_rgb = cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB)
            
            plt.figure(figsize=(12, 12))
            plt.imshow(im_array_rgb)
            plt.title(f"Inference Result - {model_name_short.upper()}")
            plt.axis('off')
            plt.show()
    else:
        print("Could not find a sample image to run inference on.")

In [ ]:
# @title 13. Process Video with All Trained Models

import os
import sys
from ultralytics import YOLO
from IPython.display import display, FileLink

# --- 1. Installation of Required Libraries ---
# This ensures ultralytics is installed in the environment.
print("✅ Checking/installing required libraries...")
!{sys.executable} -m pip install -q ultralytics
print("Libraries are ready.")


# --- 2. Define Paths and Models ---
INPUT_VIDEO_PATH = 'Wolves.mp4'
YOLOV10_VERSIONS = ['n', 's', 'm', 'l', 'x']
output_links = []

print(f"\n🎥 Input video file: '{INPUT_VIDEO_PATH}'")

if not os.path.exists(INPUT_VIDEO_PATH):
    print(f"❌ CRITICAL ERROR: Input video not found at '{INPUT_VIDEO_PATH}'")
    print("Please ensure the 'Wolves.mp4' file is uploaded to the root directory of this notebook before running.")
else:
    # --- 3. Iterate Through Each Model and Process the Video ---
    for version in YOLOV10_VERSIONS:
        model_name_short = f'yolov10{version}'
        training_run_name = f'yolo_dangerous_animal_detection_{model_name_short}'
        best_model_path = os.path.join('runs', 'detect', training_run_name, 'weights', 'best.pt')

        print(f"\n{'='*20} PROCESSING WITH {model_name_short.upper()} {'='*20}")

        if not os.path.exists(best_model_path):
            print(f"⚠️ WARNING: Model weights not found for {model_name_short.upper()} at '{best_model_path}'. Skipping this model.")
            continue

        try:
            # Load the specific trained model
            print(f"✅ Loading model: {model_name_short.upper()}...")
            model = YOLO(best_model_path)

            # Define a unique name for the output folder to keep results separate
            output_run_name = f"video_inference_{model_name_short}"
            
            # Define the desired output filename
            output_filename = f"Wolves_output_{model_name_short}.mp4"

            print(f"🚀 Starting inference on '{INPUT_VIDEO_PATH}'...")
            
            # Run prediction. The 'save=True' flag handles the video creation.
            model.predict(
                source=INPUT_VIDEO_PATH,
                save=True,
                exist_ok=True,
                conf=0.5,
                name=output_run_name
            )

            # The output is saved inside a directory named after 'output_run_name'.
            # We need to rename the file for clarity.
            original_output_path = os.path.join('runs', 'detect', output_run_name, os.path.basename(INPUT_VIDEO_PATH))
            final_output_path = os.path.join('runs', 'detect', output_run_name, output_filename)
            
            if os.path.exists(original_output_path):
                os.rename(original_output_path, final_output_path)
                print(f"✅ Inference complete for {model_name_short.upper()}!")
                print(f"   -> Saved to: {final_output_path}")
                output_links.append(final_output_path)
            else:
                 print(f"❌ ERROR: Could not find the output video file for {model_name_short.upper()} at the expected location.")

        except Exception as e:
            print(f"❌ An error occurred while processing with {model_name_short.upper()}: {e}")

    # --- 4. Display Download Links for All Processed Videos ---
    if output_links:
        print(f"\n\n{'='*20} DOWNLOAD LINKS {'='*20}")
        print("Click the links below to download your processed videos:")
        for link_path in output_links:
            print(f"\n📄 File: {os.path.basename(link_path)}")
            display(FileLink(link_path))
    else:
        print("\n\nNo videos were processed. Please check the error messages above.")